In [9]:
from pyspark.sql import *
from pyspark.sql import SparkSession
# from pyspark.ml.features import VectorAssembler
from pyspark.ml.regression import LinearRegression
from pyspark.sql.functions import col, desc
import pandas as pd
import numpy as np
from scipy import stats
import matplotlib.pyplot as plt
import seaborn as sns

In [10]:
spark = SparkSession.builder.appName("worksheetsignificance").getOrCreate()


25/12/23 11:41:56 WARN Utils: Your hostname, Biplovs-MacBook-Air.local resolves to a loopback address: 127.0.0.1; using 10.1.15.12 instead (on interface en0)
25/12/23 11:41:56 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
25/12/23 11:41:57 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
25/12/23 11:41:58 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.


In [11]:
state_df = spark.read.csv('state.csv', header = True, inferSchema = True)
state_df.show(5)

+----------+------+----------+--------+------+-------+-----+------+
|Population|Income|Illiteracy|Life Exp|Murder|HS Grad|Frost|  Area|
+----------+------+----------+--------+------+-------+-----+------+
|      3615|  3624|       2.1|   69.05|  15.1|   41.3|   20| 50708|
|       365|  6315|       1.5|   69.31|  11.3|   66.7|  152|566432|
|      2212|  4530|       1.8|   70.55|   7.8|   58.1|   15|113417|
|      2110|  3378|       1.9|   70.66|  10.1|   39.9|   65| 51945|
|     21198|  5114|       1.1|   71.71|  10.3|   62.6|   20|156361|
+----------+------+----------+--------+------+-------+-----+------+
only showing top 5 rows



In [12]:
state_pd = state_df.toPandas()
state_pd.head(5)

,Population,Income,Illiteracy,Life Exp,Murder,HS Grad,Frost,Area
0,3615,3624,2.1,69.05,15.1,41.3,20,50708
1,365,6315,1.5,69.31,11.3,66.7,152,566432
2,2212,4530,1.8,70.55,7.8,58.1,15,113417
3,2110,3378,1.9,70.66,10.1,39.9,65,51945
4,21198,5114,1.1,71.71,10.3,62.6,20,156361


In [13]:
state_pd.shape

(50, 8)

## Hypothesiss testing


In [14]:
#state_pd['Murder'].describe
#test whether the mean murder rate is 8.5
murder_data = state_pd['Murder'].values
print(murder_data)

[15.1 11.3  7.8 10.1 10.3  6.8  3.1  6.2 10.7 13.9  6.2  5.3 10.3  7.1
  2.3  4.5 10.6 13.2  2.7  8.5  3.3 11.1  2.3 12.5  9.3  5.   2.9 11.5
  3.3  5.2  9.7 10.9 11.1  1.4  7.4  6.4  4.2  6.1  2.4 11.6  1.7 11.
 12.2  4.5  5.5  9.5  4.3  6.7  3.   6.9]


In [15]:
np.mean(murder_data)

np.float64(7.377999999999999)

In [31]:
t_stat = (np.mean(murder_data) - 8.5) / (np.std(murder_data, ddof = 1) / np.sqrt(len(murder_data)))

In [17]:
df = len(murder_data)

In [38]:
p_value = 2*stats.t.sf(np.abs(t_stat), df)

In [21]:
ci = stats.t.interval(0.95, df, 
                      loc = np.mean(murder_data),
                      scale=np.std(murder_data, ddof=1) / np.sqrt(len(murder_data)))

In [23]:
print("mean:", np.mean(murder_data))
print('simple standard deviation:', np.std(murder_data))

mean: 7.377999999999999
simple standard deviation: 3.6544378500666825


In [32]:
print("Test Statistics:", t_stat)

Test Statistics: -2.1491677577323403


## imp note
test stat -2.149  
this means :
    - the simple mean 2.149 standard errors away from 8.5
    - negative sign means the sample mean iis below 8.5
    - the larger the |t|, the more unusual our sample is

|t| = 0.5 <-- very close to hypotesized mean (not unusual)

|t| = 1.5 --> moderatery far from hypothesized mean

|t| = 2.1 --> far from hypothesized mean

|t| = 5.0 --> vary far from hypothesized mean

In [36]:
# confidence interval
print("confidence interval: ",ci)

confidence interval:  (np.float64(6.3294065080918065), np.float64(8.426593491908193))


In [39]:
print("p-value", p_value)

p-value 0.03648613377210172


In [41]:
print(f"yo {'reject ho' if p_value<=0.05 else 'fail to reject ho'}")

yo reject ho
